# Projeto Redes Neurais — Experimentos com Modelos

Este notebook consolida a execução final dos experimentos com modelos do projeto. O fluxo foi organizado para carregar o artefato `datasets.pkl`, rodar os modelos tabulares e baseados em imagem com dados originais e enriquecidos, registrar métricas padronizadas por execução e salvar um CSV único ao final.

A estrutura segue a mesma ideia de organização adotada em `project_data.ipynb`: abertura, infraestrutura compartilhada, definições de modelos, execução dos experimentos e consolidação final.


In [7]:
import pickle
import time
from pathlib import Path

import mlx.core as mx
import mlx.nn as mnn
import mlx.optimizers as optim
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score
from tqdm.auto import tqdm
from xgboost import XGBClassifier

RANDOM_STATE = 42
BATCH_SIZE = 64
EPOCHS = 30
LEARNING_RATE = 0.0008
DATASETS_PICKLE_PATH = Path("data/datasets.pkl")
RESULTS_CSV_PATH = Path("data/results_models.csv")
RESULT_KEY_COLUMNS = ["model_name", "dataset_name", "input_type", "used_enriched_data"]
RESULT_COLUMNS = [
    "model_name",
    "dataset_name",
    "input_type",
    "used_enriched_data",
    "train_samples",
    "test_samples",
    "num_classes",
    "metric_average",
    "accuracy",
    "f1",
    "precision",
    "recall",
    "training_time_sec",
    "epochs",
    "image_height",
    "image_width",
]
REQUIRED_DATASET_KEYS = {
    "X_train",
    "X_test",
    "X_train_enriched",
    "y_train_enriched",
    "y_train",
    "y_test",
    "X_train_images",
    "X_train_images_enriched",
    "X_test_images",
}


## Infraestrutura compartilhada

As células abaixo carregam o artefato consolidado de dados e definem funções utilitárias reaproveitadas por todos os experimentos do notebook.


In [8]:
with open(DATASETS_PICKLE_PATH, "rb") as f:
    datasets = pickle.load(f)

for dataset_name, dataset_bundle in datasets.items():
    missing_keys = REQUIRED_DATASET_KEYS.difference(dataset_bundle.keys())
    if missing_keys:
        raise KeyError(
            f"O dataset '{dataset_name}' está sem as chaves esperadas: {sorted(missing_keys)}"
        )

print(f"datasets carregado de: {DATASETS_PICKLE_PATH}")
print("chaves disponíveis:", list(datasets.keys()))

resumo_datasets = []
for dataset_name, dataset_bundle in datasets.items():
    resumo_datasets.append(
        {
            "dataset_name": dataset_name,
            "X_train_shape": dataset_bundle["X_train"].shape,
            "X_train_enriched_shape": dataset_bundle["X_train_enriched"].shape,
            "X_test_shape": dataset_bundle["X_test"].shape,
            "X_train_images_shape": dataset_bundle["X_train_images"].shape,
            "X_train_images_enriched_shape": dataset_bundle["X_train_images_enriched"].shape,
            "X_test_images_shape": dataset_bundle["X_test_images"].shape,
        }
    )

pd.DataFrame(resumo_datasets)


datasets carregado de: data/datasets.pkl
chaves disponíveis: ['adult income', 'student', 'statlog', 'maintenance', 'stroke']


,dataset_name,X_train_shape,X_train_enriched_shape,X_test_shape,X_train_images_shape,X_train_images_enriched_shape,X_test_images_shape
0,adult income,"(39073, 12)","(59448, 12)","(9769, 12)","(39073, 24, 24)","(59448, 24, 24)","(9769, 24, 24)"
1,student,"(3539, 36)","(5301, 36)","(885, 36)","(3539, 72, 72)","(5301, 72, 72)","(885, 72, 72)"
2,statlog,"(46400, 7)","(255283, 7)","(11600, 7)","(46400, 14, 14)","(255283, 14, 14)","(11600, 14, 14)"
3,maintenance,"(8000, 6)","(15458, 6)","(2000, 6)","(8000, 12, 12)","(15458, 12, 12)","(2000, 12, 12)"
4,stroke,"(4088, 10)","(7778, 10)","(1022, 10)","(4088, 20, 20)","(7778, 20, 20)","(1022, 20, 20)"


In [9]:
def target_to_numpy(target):
    if isinstance(target, pd.Series):
        return target.to_numpy()
    return np.asarray(target)


def normalize_bool(value):
    if isinstance(value, str):
        return value.strip().lower() in {"true", "1", "yes"}
    if pd.isna(value):
        return False
    return bool(value)


def build_label_mapping(*targets):
    unique_values = set()
    for target in targets:
        unique_values.update(target_to_numpy(target).tolist())
    ordered_values = sorted(unique_values)
    return {value: index for index, value in enumerate(ordered_values)}


def encode_target(target, label_mapping):
    target_array = target_to_numpy(target)
    return np.array([label_mapping[value] for value in target_array.tolist()], dtype=np.int32)


def convert_images_to_channels_last(images):
    images_array = np.asarray(images, dtype=np.float32)
    return np.expand_dims(images_array, axis=-1) / 255.0


def compute_metrics(y_true, y_pred):
    num_classes = len(np.unique(y_true))
    average = "binary" if num_classes == 2 else "macro"

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, average=average, zero_division=0),
        "precision": precision_score(y_true, y_pred, average=average, zero_division=0),
        "recall": recall_score(y_true, y_pred, average=average, zero_division=0),
        "metric_average": average,
        "num_classes": num_classes,
    }


def build_experiment_key(model_name, dataset_name, input_type, used_enriched_data):
    return (
        str(model_name),
        str(dataset_name),
        str(input_type),
        normalize_bool(used_enriched_data),
    )


def standardize_results_df(results_df):
    standardized_df = results_df.copy()
    for column in RESULT_COLUMNS:
        if column not in standardized_df.columns:
            standardized_df[column] = pd.NA

    if not standardized_df.empty and "used_enriched_data" in standardized_df.columns:
        standardized_df["used_enriched_data"] = standardized_df["used_enriched_data"].map(normalize_bool)

    return standardized_df[RESULT_COLUMNS]


def load_results_csv(results_csv_path):
    if not results_csv_path.exists() or results_csv_path.stat().st_size == 0:
        return pd.DataFrame(columns=RESULT_COLUMNS)

    results_df = pd.read_csv(results_csv_path)
    return standardize_results_df(results_df)


def initialize_results_csv(results_csv_path):
    results_df = load_results_csv(results_csv_path)
    results_csv_path.parent.mkdir(parents=True, exist_ok=True)
    results_df.to_csv(results_csv_path, index=False)
    return results_df


def build_completed_experiment_keys(results_df):
    if results_df.empty:
        return set()

    missing_columns = [column for column in RESULT_KEY_COLUMNS if column not in results_df.columns]
    if missing_columns:
        raise KeyError(
            f"O CSV de resultados está sem as colunas de chave esperadas: {missing_columns}"
        )

    completed_keys = set()
    for _, row in results_df.iterrows():
        completed_keys.add(
            build_experiment_key(
                row["model_name"],
                row["dataset_name"],
                row["input_type"],
                row["used_enriched_data"],
            )
        )

    return completed_keys


def experiment_exists(
    completed_experiment_keys,
    *,
    model_name,
    dataset_name,
    input_type,
    used_enriched_data,
):
    experiment_key = build_experiment_key(
        model_name,
        dataset_name,
        input_type,
        used_enriched_data,
    )
    return experiment_key in completed_experiment_keys


def print_skip_message(model_name, dataset_name, input_type, used_enriched_data):
    origem = "enriched" if normalize_bool(used_enriched_data) else "original"
    print(
        f"Pulando experimento já concluído: modelo={model_name} | dataset={dataset_name} | "
        f"entrada={input_type} | dados={origem}"
    )


def register_result(
    results,
    completed_experiment_keys,
    results_csv_path,
    *,
    model_name,
    dataset_name,
    input_type,
    used_enriched_data,
    train_samples,
    test_samples,
    training_time_sec,
    metrics,
    extra=None,
):
    row = {column: None for column in RESULT_COLUMNS}
    row.update(
        {
            "model_name": model_name,
            "dataset_name": dataset_name,
            "input_type": input_type,
            "used_enriched_data": normalize_bool(used_enriched_data),
            "train_samples": train_samples,
            "test_samples": test_samples,
            "num_classes": metrics["num_classes"],
            "metric_average": metrics["metric_average"],
            "accuracy": metrics["accuracy"],
            "f1": metrics["f1"],
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "training_time_sec": training_time_sec,
        }
    )

    if extra:
        for key, value in extra.items():
            if key in row:
                row[key] = value

    experiment_key = build_experiment_key(
        row["model_name"],
        row["dataset_name"],
        row["input_type"],
        row["used_enriched_data"],
    )
    if experiment_key in completed_experiment_keys:
        print_skip_message(
            row["model_name"],
            row["dataset_name"],
            row["input_type"],
            row["used_enriched_data"],
        )
        return False

    row_df = pd.DataFrame([row], columns=RESULT_COLUMNS)
    results_csv_path.parent.mkdir(parents=True, exist_ok=True)
    write_header = not results_csv_path.exists() or results_csv_path.stat().st_size == 0
    row_df.to_csv(results_csv_path, mode="a", header=write_header, index=False)

    results.append(row)
    completed_experiment_keys.add(experiment_key)
    return True


def print_experiment_summary(model_name, dataset_name, input_type, used_enriched_data, metrics, training_time_sec):
    origem = "enriched" if normalize_bool(used_enriched_data) else "original"
    print("-" * 80)
    print(
        f"Modelo: {model_name} | Dataset: {dataset_name} | Entrada: {input_type} | Dados: {origem}"
    )
    print(f"Accuracy: {metrics['accuracy']:.4f}")
    print(f"F1: {metrics['f1']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall: {metrics['recall']:.4f}")
    print(f"Tempo de treinamento: {training_time_sec:.2f}s")


## Definições de modelos

A arquitetura base da CNN e as configurações dos modelos ficam centralizadas aqui para evitar repetição nas células de execução.


In [10]:
def estimate_conv_output_size(size, stride):
    return int(np.ceil(size / stride))


def compute_pooling_targets(image_height, image_width):
    base_targets = [(15, 15), (7, 7), (3, 3), (1, 1)]
    strides = [1, 2, 2, 2]

    current_height = int(image_height)
    current_width = int(image_width)
    if current_height < 1 or current_width < 1:
        raise ValueError(
            f"Tamanho de imagem inválido para a CNN: {(image_height, image_width)}"
        )

    pooling_targets = []
    for (base_height, base_width), stride in zip(base_targets, strides):
        conv_height = estimate_conv_output_size(current_height, stride)
        conv_width = estimate_conv_output_size(current_width, stride)

        target_height = max(1, min(base_height, conv_height))
        target_width = max(1, min(base_width, conv_width))

        if target_height > conv_height or target_width > conv_width:
            raise ValueError(
                "Pooling target inválido calculado para a CNN: "
                f"entrada={(current_height, current_width)}, conv={(conv_height, conv_width)}, "
                f"target={(target_height, target_width)}"
            )

        pooling_targets.append((target_height, target_width))
        current_height, current_width = target_height, target_width

    return pooling_targets


def adaptive_max_pool2d(x, output_size):
    in_height, in_width = int(x.shape[1]), int(x.shape[2])
    out_height = min(int(output_size[0]), in_height)
    out_width = min(int(output_size[1]), in_width)

    if out_height < 1 or out_width < 1:
        raise ValueError(
            f"Adaptive pooling recebeu target inválido {output_size} para entrada {(in_height, in_width)}"
        )

    stride_height = max(1, in_height // out_height)
    stride_width = max(1, in_width // out_width)
    kernel_height = max(1, in_height - (out_height - 1) * stride_height)
    kernel_width = max(1, in_width - (out_width - 1) * stride_width)

    pool = mnn.MaxPool2d(
        kernel_size=(kernel_height, kernel_width),
        stride=(stride_height, stride_width),
        padding=0,
    )
    return pool(x)


class SEBlock(mnn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(1, channels // reduction)
        self.fc1 = mnn.Linear(channels, hidden)
        self.fc2 = mnn.Linear(hidden, channels)

    def __call__(self, x):
        weights = mx.mean(x, axis=(1, 2))
        weights = mnn.relu(self.fc1(weights))
        weights = mx.sigmoid(self.fc2(weights))
        weights = mx.reshape(weights, (x.shape[0], 1, 1, x.shape[-1]))
        return x * weights


class ConvBlock(mnn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        stride,
        target_size,
        use_se=False,
        use_asymmetric=False,
        use_dilation=False,
        use_residual=False,
    ):
        super().__init__()
        self.target_size = target_size
        self.use_se = use_se
        self.use_asymmetric = use_asymmetric
        self.use_residual = use_residual

        dilation = 2 if use_dilation else 1
        padding = dilation

        if use_asymmetric:
            self.conv_a = mnn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=(1, 3),
                stride=(1, stride),
                padding=(0, padding),
                dilation=(1, dilation),
            )
            self.conv_b = mnn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=(3, 1),
                stride=(stride, 1),
                padding=(padding, 0),
                dilation=(dilation, 1),
            )
        else:
            self.conv = mnn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                stride=stride,
                padding=padding,
                dilation=dilation,
            )

        self.bn = mnn.BatchNorm(out_channels)
        self.se = SEBlock(out_channels) if use_se else None

        needs_projection = use_residual and (in_channels != out_channels or stride != 1)
        self.proj = (
            mnn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride)
            if needs_projection
            else None
        )

    def __call__(self, x):
        shortcut = x

        if self.use_asymmetric:
            x = self.conv_a(x)
            x = mnn.relu(x)
            x = self.conv_b(x)
        else:
            x = self.conv(x)

        x = self.bn(x)
        x = mnn.relu(x)

        if self.se is not None:
            x = self.se(x)

        if self.use_residual:
            if self.proj is not None:
                shortcut = self.proj(shortcut)
            x = x + shortcut

        x = adaptive_max_pool2d(x, self.target_size)
        return x


class NCTD_CNN_Base(mnn.Module):
    def __init__(
        self,
        num_classes,
        input_shape=None,
        use_se=False,
        use_asymmetric=False,
        use_dilation=False,
        use_residual=False,
    ):
        super().__init__()

        if input_shape is None:
            pooling_targets = [(15, 15), (7, 7), (3, 3), (1, 1)]
        else:
            if len(input_shape) != 2:
                raise ValueError(f"input_shape inválido para a CNN: {input_shape}")
            pooling_targets = compute_pooling_targets(input_shape[0], input_shape[1])

        self.pooling_targets = pooling_targets

        self.block0 = ConvBlock(
            1,
            64,
            stride=1,
            target_size=pooling_targets[0],
            use_se=use_se,
            use_asymmetric=use_asymmetric,
            use_dilation=False,
            use_residual=use_residual,
        )
        self.block1 = ConvBlock(
            64,
            64,
            stride=2,
            target_size=pooling_targets[1],
            use_se=use_se,
            use_asymmetric=use_asymmetric,
            use_dilation=use_dilation,
            use_residual=use_residual,
        )
        self.block2 = ConvBlock(
            64,
            64,
            stride=2,
            target_size=pooling_targets[2],
            use_se=use_se,
            use_asymmetric=use_asymmetric,
            use_dilation=use_dilation,
            use_residual=use_residual,
        )
        self.block3 = ConvBlock(
            64,
            64,
            stride=2,
            target_size=pooling_targets[3],
            use_se=use_se,
            use_asymmetric=use_asymmetric,
            use_dilation=False,
            use_residual=use_residual,
        )

        self.fc = mnn.Linear(64, 32)
        self.out = mnn.Linear(32, num_classes)

    def __call__(self, x):
        x = self.block0(x)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = mx.mean(x, axis=(1, 2))
        x = mnn.relu(self.fc(x))
        x = self.out(x)
        return x


CNN_MODEL_BUILDERS = [
    (
        "NCTD Base",
        lambda num_classes, input_shape: NCTD_CNN_Base(
            num_classes=num_classes,
            input_shape=input_shape,
        ),
    ),
    (
        "NCTD Attention (SE)",
        lambda num_classes, input_shape: NCTD_CNN_Base(
            num_classes=num_classes,
            input_shape=input_shape,
            use_se=True,
        ),
    ),
    (
        "NCTD Asymmetric Kernels",
        lambda num_classes, input_shape: NCTD_CNN_Base(
            num_classes=num_classes,
            input_shape=input_shape,
            use_asymmetric=True,
        ),
    ),
    (
        "NCTD Dilated Convolutions",
        lambda num_classes, input_shape: NCTD_CNN_Base(
            num_classes=num_classes,
            input_shape=input_shape,
            use_dilation=True,
        ),
    ),
    (
        "NCTD Residual",
        lambda num_classes, input_shape: NCTD_CNN_Base(
            num_classes=num_classes,
            input_shape=input_shape,
            use_residual=True,
        ),
    ),
]


## Funções de execução dos experimentos

As rotinas abaixo recebem explicitamente treino e teste de cada dataset, executam o modelo correspondente e registram os resultados em uma estrutura única.


In [11]:
def run_xgboost_experiment(
    *,
    dataset_name,
    X_train,
    y_train,
    X_test,
    y_test,
    used_enriched_data,
    results,
    completed_experiment_keys,
    results_csv_path,
):
    label_mapping = build_label_mapping(y_train, y_test)
    y_train_encoded = encode_target(y_train, label_mapping)
    y_test_encoded = encode_target(y_test, label_mapping)
    num_classes = len(label_mapping)

    model_params = {
        "n_estimators": 100,
        "learning_rate": 0.1,
        "max_depth": 6,
        "random_state": RANDOM_STATE,
    }

    if num_classes == 2:
        model_params.update({"objective": "binary:logistic", "eval_metric": "logloss"})
    else:
        model_params.update(
            {
                "objective": "multi:softprob",
                "num_class": num_classes,
                "eval_metric": "mlogloss",
            }
        )

    model = XGBClassifier(**model_params)

    start_time = time.perf_counter()
    model.fit(X_train, y_train_encoded)
    training_time_sec = time.perf_counter() - start_time

    y_pred = model.predict(X_test)
    metrics = compute_metrics(y_test_encoded, y_pred)

    print_experiment_summary(
        "XGBoost",
        dataset_name,
        "tabular",
        used_enriched_data,
        metrics,
        training_time_sec,
    )
    print(classification_report(y_test_encoded, y_pred, zero_division=0))

    return register_result(
        results,
        completed_experiment_keys,
        results_csv_path,
        model_name="XGBoost",
        dataset_name=dataset_name,
        input_type="tabular",
        used_enriched_data=used_enriched_data,
        train_samples=len(X_train),
        test_samples=len(X_test),
        training_time_sec=training_time_sec,
        metrics=metrics,
    )


def run_cnn_experiment(
    *,
    model_name,
    model_builder,
    dataset_name,
    X_train_images,
    y_train,
    X_test_images,
    y_test,
    used_enriched_data,
    results,
    completed_experiment_keys,
    results_csv_path,
    epochs=EPOCHS,
):
    train_images_array = np.asarray(X_train_images)
    test_images_array = np.asarray(X_test_images)

    if train_images_array.ndim != 3 or test_images_array.ndim != 3:
        raise ValueError(
            f"As imagens da CNN devem ter shape (N, H, W). "
            f"Recebido treino={train_images_array.shape} e teste={test_images_array.shape} "
            f"para o dataset '{dataset_name}'."
        )

    input_shape = tuple(train_images_array.shape[1:3])
    if tuple(test_images_array.shape[1:3]) != input_shape:
        raise ValueError(
            f"Shapes de imagem incompatíveis entre treino e teste para o dataset '{dataset_name}': "
            f"treino={input_shape}, teste={tuple(test_images_array.shape[1:3])}"
        )

    pooling_targets = compute_pooling_targets(*input_shape)

    label_mapping = build_label_mapping(y_train, y_test)
    y_train_encoded = encode_target(y_train, label_mapping)
    y_test_encoded = encode_target(y_test, label_mapping)
    num_classes = len(label_mapping)
    if num_classes < 2:
        raise ValueError(
            f"A CNN exige pelo menos 2 classes. Dataset '{dataset_name}' retornou {num_classes}."
        )

    X_train_mlx = convert_images_to_channels_last(train_images_array)
    X_test_mlx = convert_images_to_channels_last(test_images_array)

    model = model_builder(num_classes, input_shape)
    mx.eval(model.parameters())
    optimizer = optim.Adam(learning_rate=LEARNING_RATE)

    def loss_fn(model, x, y):
        logits = model(x)
        return mx.mean(mnn.losses.cross_entropy(logits, y))

    loss_and_grad_fn = mnn.value_and_grad(model, loss_fn)
    rng = np.random.default_rng(RANDOM_STATE)

    num_batches_train = (len(X_train_mlx) + BATCH_SIZE - 1) // BATCH_SIZE
    num_batches_test = (len(X_test_mlx) + BATCH_SIZE - 1) // BATCH_SIZE
    total_steps = epochs * num_batches_train + num_batches_test
    origem = "enriched" if used_enriched_data else "original"

    print(f"Pooling targets para {dataset_name}/{model_name}: {pooling_targets}")
    print(f"Número de classes na saída: {num_classes}")

    start_time = time.perf_counter()

    with tqdm(
        total=total_steps,
        desc=f"{dataset_name} | {model_name} | {origem}",
        unit="batch",
    ) as pbar:
        for epoch in range(epochs):
            model.train()
            running_loss = 0.0
            permutation = rng.permutation(len(X_train_mlx))

            for start in range(0, len(X_train_mlx), BATCH_SIZE):
                end = min(start + BATCH_SIZE, len(X_train_mlx))
                batch_indices = permutation[start:end]

                batch_x = mx.array(X_train_mlx[batch_indices])
                batch_y = mx.array(y_train_encoded[batch_indices])

                loss, grads = loss_and_grad_fn(model, batch_x, batch_y)
                optimizer.update(model, grads)
                mx.eval(model.parameters(), optimizer.state)

                loss_value = loss.item()
                running_loss += loss_value * len(batch_indices)

                pbar.update(1)
                pbar.set_postfix(
                    {
                        "epoch": f"{epoch + 1}/{epochs}",
                        "loss": f"{loss_value:.4f}",
                    }
                )

            epoch_loss = running_loss / len(X_train_mlx)
            pbar.set_postfix(
                {
                    "epoch": f"{epoch + 1}/{epochs}",
                    "loss": f"{epoch_loss:.4f}",
                }
            )

        model.eval()
        predictions = []

        for start in range(0, len(X_test_mlx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(X_test_mlx))
            batch_x = mx.array(X_test_mlx[start:end])

            logits = model(batch_x)
            preds = mx.argmax(logits, axis=1)
            mx.eval(preds)

            predictions.extend(np.array(preds).tolist())
            pbar.update(1)
            pbar.set_postfix({"fase": "teste"})

    training_time_sec = time.perf_counter() - start_time
    predictions = np.array(predictions, dtype=np.int32)
    metrics = compute_metrics(y_test_encoded, predictions)

    print_experiment_summary(
        model_name,
        dataset_name,
        "image",
        used_enriched_data,
        metrics,
        training_time_sec,
    )
    print(classification_report(y_test_encoded, predictions, zero_division=0))

    return register_result(
        results,
        completed_experiment_keys,
        results_csv_path,
        model_name=model_name,
        dataset_name=dataset_name,
        input_type="image",
        used_enriched_data=used_enriched_data,
        train_samples=len(X_train_mlx),
        test_samples=len(X_test_mlx),
        training_time_sec=training_time_sec,
        metrics=metrics,
        extra={
            "epochs": epochs,
            "image_height": input_shape[0],
            "image_width": input_shape[1],
        },
    )


## Execução dos experimentos

Cada dataset é processado com o mesmo protocolo: XGBoost para dados tabulares e as variantes finais da NCTD para dados em imagem, sempre comparando treino original e treino enriquecido contra o mesmo conjunto de teste.


In [12]:
existing_results_df = initialize_results_csv(RESULTS_CSV_PATH)
completed_experiment_keys = build_completed_experiment_keys(existing_results_df)

results = []
executed_in_session = 0
skipped_in_session = 0

print(f"Resultados já persistidos no CSV: {len(completed_experiment_keys)}")

for dataset_name, dataset_bundle in datasets.items():
    print("\n" + "=" * 80)
    print(f"Iniciando experimentos para o dataset: {dataset_name}")
    print("=" * 80)

    if experiment_exists(
        completed_experiment_keys,
        model_name="XGBoost",
        dataset_name=dataset_name,
        input_type="tabular",
        used_enriched_data=False,
    ):
        skipped_in_session += 1
        print_skip_message("XGBoost", dataset_name, "tabular", False)
    else:
        executed_in_session += int(
            run_xgboost_experiment(
                dataset_name=dataset_name,
                X_train=dataset_bundle["X_train"],
                y_train=dataset_bundle["y_train"],
                X_test=dataset_bundle["X_test"],
                y_test=dataset_bundle["y_test"],
                used_enriched_data=False,
                results=results,
                completed_experiment_keys=completed_experiment_keys,
                results_csv_path=RESULTS_CSV_PATH,
            )
        )

    if experiment_exists(
        completed_experiment_keys,
        model_name="XGBoost",
        dataset_name=dataset_name,
        input_type="tabular",
        used_enriched_data=True,
    ):
        skipped_in_session += 1
        print_skip_message("XGBoost", dataset_name, "tabular", True)
    else:
        executed_in_session += int(
            run_xgboost_experiment(
                dataset_name=dataset_name,
                X_train=dataset_bundle["X_train_enriched"],
                y_train=dataset_bundle["y_train_enriched"],
                X_test=dataset_bundle["X_test"],
                y_test=dataset_bundle["y_test"],
                used_enriched_data=True,
                results=results,
                completed_experiment_keys=completed_experiment_keys,
                results_csv_path=RESULTS_CSV_PATH,
            )
        )

    for model_name, model_builder in CNN_MODEL_BUILDERS:
        if experiment_exists(
            completed_experiment_keys,
            model_name=model_name,
            dataset_name=dataset_name,
            input_type="image",
            used_enriched_data=False,
        ):
            skipped_in_session += 1
            print_skip_message(model_name, dataset_name, "image", False)
        else:
            executed_in_session += int(
                run_cnn_experiment(
                    model_name=model_name,
                    model_builder=model_builder,
                    dataset_name=dataset_name,
                    X_train_images=dataset_bundle["X_train_images"],
                    y_train=dataset_bundle["y_train"],
                    X_test_images=dataset_bundle["X_test_images"],
                    y_test=dataset_bundle["y_test"],
                    used_enriched_data=False,
                    results=results,
                    completed_experiment_keys=completed_experiment_keys,
                    results_csv_path=RESULTS_CSV_PATH,
                )
            )

        if experiment_exists(
            completed_experiment_keys,
            model_name=model_name,
            dataset_name=dataset_name,
            input_type="image",
            used_enriched_data=True,
        ):
            skipped_in_session += 1
            print_skip_message(model_name, dataset_name, "image", True)
        else:
            executed_in_session += int(
                run_cnn_experiment(
                    model_name=model_name,
                    model_builder=model_builder,
                    dataset_name=dataset_name,
                    X_train_images=dataset_bundle["X_train_images_enriched"],
                    y_train=dataset_bundle["y_train_enriched"],
                    X_test_images=dataset_bundle["X_test_images"],
                    y_test=dataset_bundle["y_test"],
                    used_enriched_data=True,
                    results=results,
                    completed_experiment_keys=completed_experiment_keys,
                    results_csv_path=RESULTS_CSV_PATH,
                )
            )

results_df = load_results_csv(RESULTS_CSV_PATH)
print(f"Experimentos executados nesta sessão: {executed_in_session}")
print(f"Experimentos pulados por já existirem: {skipped_in_session}")
print(f"Total persistido no CSV: {len(results_df)}")
results_df.head()


Resultados já persistidos no CSV: 0

Iniciando experimentos para o dataset: adult income
--------------------------------------------------------------------------------
Modelo: XGBoost | Dataset: adult income | Entrada: tabular | Dados: original
Accuracy: 0.8772
F1: 0.7167
Precision: 0.7998
Recall: 0.6493
Tempo de treinamento: 0.22s
              precision    recall  f1-score   support

           0       0.90      0.95      0.92      7431
           1       0.80      0.65      0.72      2338

    accuracy                           0.88      9769
   macro avg       0.85      0.80      0.82      9769
weighted avg       0.87      0.88      0.87      9769

--------------------------------------------------------------------------------
Modelo: XGBoost | Dataset: adult income | Entrada: tabular | Dados: enriched
Accuracy: 0.8747
F1: 0.7083
Precision: 0.7998
Recall: 0.6356
Tempo de treinamento: 0.19s
              precision    recall  f1-score   support

           0       0.89      0.95  

adult income | NCTD Base | original: 100%|██████████| 18483/18483 [06:58<00:00, 44.13batch/s, fase=teste]              


--------------------------------------------------------------------------------
Modelo: NCTD Base | Dataset: adult income | Entrada: image | Dados: original
Accuracy: 0.8493
F1: 0.6214
Precision: 0.7794
Recall: 0.5167
Tempo de treinamento: 418.84s
              precision    recall  f1-score   support

           0       0.86      0.95      0.91      7431
           1       0.78      0.52      0.62      2338

    accuracy                           0.85      9769
   macro avg       0.82      0.74      0.76      9769
weighted avg       0.84      0.85      0.84      9769

Pooling targets para adult income/NCTD Base: [(15, 15), (7, 7), (3, 3), (1, 1)]
Número de classes na saída: 2


adult income | NCTD Base | enriched: 100%|██████████| 28023/28023 [10:11<00:00, 45.82batch/s, fase=teste]              


--------------------------------------------------------------------------------
Modelo: NCTD Base | Dataset: adult income | Entrada: image | Dados: enriched
Accuracy: 0.8427
F1: 0.6303
Precision: 0.7202
Recall: 0.5603
Tempo de treinamento: 611.55s
              precision    recall  f1-score   support

           0       0.87      0.93      0.90      7431
           1       0.72      0.56      0.63      2338

    accuracy                           0.84      9769
   macro avg       0.80      0.75      0.77      9769
weighted avg       0.83      0.84      0.84      9769

Pooling targets para adult income/NCTD Attention (SE): [(15, 15), (7, 7), (3, 3), (1, 1)]
Número de classes na saída: 2


...

stroke | NCTD Residual | original: 100%|██████████| 1936/1936 [00:22<00:00, 85.79batch/s, fase=teste]              


--------------------------------------------------------------------------------
Modelo: NCTD Residual | Dataset: stroke | Entrada: image | Dados: original
Accuracy: 0.9511
F1: 0.0000
Precision: 0.0000
Recall: 0.0000
Tempo de treinamento: 22.57s
              precision    recall  f1-score   support

           0       0.95      1.00      0.97       972
           1       0.00      0.00      0.00        50

    accuracy                           0.95      1022
   macro avg       0.48      0.50      0.49      1022
weighted avg       0.90      0.95      0.93      1022

Pooling targets para stroke/NCTD Residual: [(15, 15), (7, 7), (3, 3), (1, 1)]
Número de classes na saída: 2


stroke | NCTD Residual | enriched: 100%|██████████| 3676/3676 [00:42<00:00, 85.73batch/s, fase=teste]              


--------------------------------------------------------------------------------
Modelo: NCTD Residual | Dataset: stroke | Entrada: image | Dados: enriched
Accuracy: 0.8796
F1: 0.1854
Precision: 0.1386
Recall: 0.2800
Tempo de treinamento: 42.88s
              precision    recall  f1-score   support

           0       0.96      0.91      0.94       972
           1       0.14      0.28      0.19        50

    accuracy                           0.88      1022
   macro avg       0.55      0.60      0.56      1022
weighted avg       0.92      0.88      0.90      1022

Experimentos executados nesta sessão: 60
Experimentos pulados por já existirem: 0
Total persistido no CSV: 60


,model_name,dataset_name,input_type,used_enriched_data,train_samples,test_samples,num_classes,metric_average,accuracy,f1,precision,recall,training_time_sec,epochs,image_height,image_width
0,XGBoost,adult income,tabular,False,39073,9769,2,binary,0.877162,0.716714,0.799789,0.649273,0.222576,NaN,NaN,NaN
1,XGBoost,adult income,tabular,True,59448,9769,2,binary,0.874706,0.708294,0.799785,0.635586,0.194354,NaN,NaN,NaN
2,NCTD Base,adult income,image,False,39073,9769,2,binary,0.849319,0.621399,0.779355,0.516681,418.838451,30.0,24.0,24.0
3,NCTD Base,adult income,image,True,59448,9769,2,binary,0.842666,0.630262,0.720176,0.560308,611.552244,30.0,24.0,24.0
4,NCTD Attention (SE),adult income,image,False,39073,9769,2,binary,0.849524,0.631209,0.763350,0.538067,422.542605,30.0,24.0,24.0


## Consolidação final

Ao final da execução, todos os resultados são reunidos em um único DataFrame e persistidos em CSV para análise posterior em outro notebook.


In [13]:
results_df = load_results_csv(RESULTS_CSV_PATH)

if results_df.empty:
    resumo_final = results_df
else:
    resumo_final = results_df.sort_values(
        ["dataset_name", "input_type", "model_name", "used_enriched_data"]
    ).reset_index(drop=True)

print(f"Resultados persistidos em: {RESULTS_CSV_PATH}")
print(f"Execuções novas nesta sessão: {executed_in_session}")
print(f"Execuções puladas nesta sessão: {skipped_in_session}")

resumo_final


Resultados persistidos em: data/results_models.csv
Execuções novas nesta sessão: 60
Execuções puladas nesta sessão: 0


,model_name,dataset_name,input_type,used_enriched_data,train_samples,test_samples,num_classes,metric_average,accuracy,f1,precision,recall,training_time_sec,epochs,image_height,image_width
0,NCTD Asymmetric Kernels,adult income,image,False,39073,9769,2,binary,0.850957,0.672220,0.709601,0.638580,435.515064,30.0,24.0,24.0
1,NCTD Asymmetric Kernels,adult income,image,True,59448,9769,2,binary,0.813594,0.630104,0.600000,0.663388,679.711279,30.0,24.0,24.0
2,NCTD Attention (SE),adult income,image,False,39073,9769,2,binary,0.849524,0.631209,0.763350,0.538067,422.542605,30.0,24.0,24.0
3,NCTD Attention (SE),adult income,image,True,59448,9769,2,binary,0.851264,0.646558,0.749577,0.568435,654.945175,30.0,24.0,24.0
4,NCTD Base,adult income,image,False,39073,9769,2,binary,0.849319,0.621399,0.779355,0.516681,418.838451,30.0,24.0,24.0
5,NCTD Base,adult income,image,True,59448,9769,2,binary,0.842666,0.630262,0.720176,0.560308,611.552244,30.0,24.0,24.0
6,NCTD Dilated Convolutions,adult income,image,False,39073,9769,2,binary,0.848500,0.632389,0.754147,0.544482,417.589121,30.0,24.0,24.0
7,NCTD Dilated Convolutions,adult income,image,True,59448,9769,2,binary,0.844406,0.646512,0.708461,0.594525,598.417570,30.0,24.0,24.0
8,NCTD Residual,adult income,image,False,39073,9769,2,binary,0.846351,0.637001,0.732888,0.563302,417.563087,30.0,24.0,24.0
9,NCTD Residual,adult income,image,True,59448,9769,2,binary,0.848091,0.646667,0.729323,0.580838,638.513975,30.0,24.0,24.0
